In [0]:

create or replace temporary view base as
with max_date as (
  select coalesce(max(trade_date), '1900-01-01') as max_trade_date
  from analytics.stock_daily_metrics
)
select 
  f.ticker,
  f.trade_date,
  f.close,
  lag(f.close) over (
    partition by f.ticker
    order by f.trade_date
  ) as prev_close,
  avg(f.close) over (
    partition by f.ticker
    order by f.trade_date
    rows between 9 preceding and current row
  ) as ma_10,
  avg(f.close) over (
    partition by f.ticker
    order by f.trade_date
    rows between 19 preceding and current row
  ) as ma_20,
  avg(f.close) over (
    partition by f.ticker
    order by f.trade_date
    rows between 49 preceding and current row
  ) as ma_50
from analytics.fact_stock_daily f
cross join max_date m
where f.trade_date >= date_sub(m.max_trade_date, 50);

merge into analytics.stock_daily_metrics target
using (
  select 
    ticker,
    trade_date,
    close,
    (close - prev_close) / nullif(prev_close, 0) as daily_return,
    ma_10,
    ma_20,
    ma_50
   from base
) source
on target.ticker = source.ticker
and target.trade_date = source.trade_date
when matched then update set *
when not matched then insert *